1. 문서의 내용으 읽는다
2. 문서를 쪼갠다.
  - 토큰 수 초과로 답변을 생성하지 못할 수 있고
  - 문서가 길면(Input) 답변 생성이 오래걸림
3. 쪼갠 문서를 임배딩 -> 백터 데이터베이스에 저장
4. 질문이 있을 때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# %pip install --upgrade --quiet  docx2txt langchain-community


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 문서 쪼개기 Text Splitter

# Character Text Splitter : 구분자를 하나만 넣을 수 있다.
# Recursive Character Text Splitter : 구분자를 여러개 넣을 수 있다.

# %pip install -qU langchain-text-splitters







Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
print(os.path.exists("./tax.docx"))

True


In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # 문서를 쪼갤 때 하나의 청크가 가질 수 있는 청크의 수
    chunk_size=1500,
    # 문서를 곂치는 설정
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax_with_markdown.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
len(document_list)

193

In [3]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings


load_dotenv()

embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')


In [6]:
# %pip install langchain-chroma

In [4]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-openai \
#     langchain \
#     pinecone-notebooks
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-table-index'
pinecone_api_key= os.environ.get("PINECONE_API_KEY")
pinecone_api_key

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding=embedding, index_name=index_name)


In [39]:
query = '연봉 5천만원인 직장인의 소득세는?'
retrieved_docs = database.similarity_search(query, k=3)

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')



In [28]:
prompt = f"""
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question : {query}
"""

In [29]:
ai_message = llm.invoke(prompt)

ai_message.content




'연봉 5천만원인 거주자의 소득세를 계산하기 위해서는 소득세 과세표준에 따른 세율을 적용해야 합니다. [Context]에 제공된 소득세율을 참조하여 계산해보겠습니다.\n\n1. 연봉 5천만원에 대한 소득세율을 확인합니다:\n   - 1,400만원 초과 5,000만원 이하는 84만원 + (1,400만원을 초과하는 금액의 15%)\n\n2. 5,000만원 - 1,400만원 = 3,600만원 (1,400만원을 초과하는 금액)\n\n3. 초과 금액에 대한 세금: \n   - 3,600만원의 15% = 540만원\n\n4. 전체 소득세:\n   - 84만원 + 540만원 = 624만원\n\n따라서, 연봉 5천만원인 거주자의 소득세는 624만원이 됩니다.'

In [12]:
# %pip install -U langchain langchain_classic langchainhub --quiet

In [30]:
from langchain_classic import hub

prompt = hub.pull('rlm/rag-prompt')


In [14]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [31]:
from langchain_classic.chains import RetrievalQA

retriever=database.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type_kwargs={'prompt': prompt}
)

In [32]:
retriever.invoke(query)

[Document(id='3721ecd1-0606-422d-933f-6fe79031d547', metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소

In [33]:
ai_message = qa_chain.invoke({'query': query})

In [34]:
ai_message

{'query': '연봉 5천만원인 거주자의 소득세는?',
 'result': '연봉 5천만 원인 거주자의 소득세는 624만 원입니다. 이 금액은 종합소득 과세표준이 5천만 원 초과 8,800만 원 이하 구간에 속하므로, 5천만 원 초과분에 대한 24%의 세율을 기본 624만 원에 더한다는 규칙을 따릅니다.'}

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단되다면, 사용자의 질문을 변경하지 않아도 됩니다.
    변경하지 않아도 되는 경우에는 질문만 리턴해주세요.
    
    사전 : {dictionary}

    질문 : {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()

In [41]:
new_question = dictionary_chain.invoke({"question" : query })

In [43]:
query

'연봉 5천만원인 직장인의 소득세는?'

In [42]:
new_question

'연봉 5천만원인 거주자의 소득세는?'

In [44]:
tax_chain = {"query" : dictionary_chain} | qa_chain

In [45]:
ai_response = tax_chain.invoke({"question" : query})

In [46]:
ai_response

{'query': '연봉 5천만원인 거주자의 소득세는?',
 'result': '연봉 5천만원인 거주자의 소득세는 과세표준 범위에 따라 계산됩니다. 5천만원은 1,400만원 초과 5,000만원 이하 구간에 해당하며, 소득세는 84만원에 1,400만원을 초과하는 금액의 15%를 더하여 계산됩니다. 이는 총 84만원 + (3,600만원 * 15%) = 624만원이 됩니다.'}